In [221]:
class MinHeap:
  def __init__(self, cap):
    self.arr = [0]*cap
    self.size = 0
    self.cap = cap

  def insert(self, node):
    if self.size < self.cap:
      self.size += 1
      self.arr[self.size-1] = node
      self.swim(self.size-1)
    else:
      return

  def swim(self, index):
    if index>0:
      parent = (index-1)//2
      if self.arr[index].distance < self.arr[parent].distance:
        self.arr[index], self.arr[parent] = self.arr[parent], self.arr[index]
        self.swim(parent)
      else:
        return

  def extractMin(self):
    temp = self.arr[0]
    self.arr[0], self.arr[self.size-1] = self.arr[self.size-1], self.arr[0]
    self.size-=1
    self.arr[self.size] = 0
    self.sink(0)
    return temp

  def sink(self, index):
    if index<self.size:
      left = (2*index)+1
      right = (2*index)+2
      if left < self.size:
        if right < self.size:
          small = 0
          if self.arr[left].distance < self.arr[right].distance:
            small = left
          else:
            small = right
          if self.arr[small].distance < self.arr[index].distance:
            self.arr[small], self.arr[index] = self.arr[index], self.arr[small]
            self.sink(small)
        else:
          if self.arr[left].distance < self.arr[index].distance:
            self.arr[left], self.arr[index] = self.arr[index], self.arr[left]
            self.sink(left)
    return

  def get_index(self, node):
      for i in range(self.size):
        if self.arr[i].key == node:
          return i

In [222]:
class Vertex:
    def __init__(self, key):
        self.key = key
        self.color = "white"
        self.parent = None
        self.distance = 0

In [223]:
class Graph:
    def __init__(self):
        # Map keys to Vertex objects.
        self.vertices = {}
        # Adjacency list: mapping vertex key -> list of adjacent Vertex objects.
        self.adj_list = {}

    def add_vertex(self, key):
        v = Vertex(key)
        self.vertices[key] = v
        self.adj_list[key] = []

    def add_edge(self, from_key, to_key, weight, directed=True):
        self.adj_list[from_key].append((self.vertices[to_key], weight))
        if directed==False:
          self.adj_list[to_key].append((self.vertices[from_key], weight))


## Dijkstra's Algorithm

In [224]:
def dijkstra(graph, source):
  for key in graph.vertices:
    u = graph.vertices[key]
    u.distance = float('inf')

  graph.vertices[source].distance = 0
  heap = MinHeap(len(graph.vertices))
  for key in graph.vertices:
    node = graph.vertices[key]
    heap.insert(node)

  while heap.size>0:
    u_node = heap.extractMin()
    u_node.color = "black"
    for v in graph.adj_list[u_node.key]:
      v_node, v_weight = v[0], v[1]
      if v_node.color=='white':
        if u_node.distance + v_weight < v_node.distance:
          v_node.distance = u_node.distance + v_weight
          v_node.parent = u_node.key
          idx = heap.get_index(v_node.key)
          heap.swim(idx)

In [225]:
g = Graph()
for key in ["S", "A", "B", "C", "D"]:
  g.add_vertex(key)

g.add_edge("S", "A", 10)
g.add_edge("S", "C", 5)

g.add_edge("A", "B", 1)
g.add_edge("A", "C", 2)

g.add_edge("B", "D", 4)

g.add_edge("C", "A", 3)
g.add_edge("C", "B", 9)
g.add_edge("C", "D", 2)

g.add_edge("D", "B", 6)
g.add_edge("D", "S", 7)

dijkstra(g, "S")

In [226]:
for key, v in g.vertices.items():
        print(f"{key}: distance = {v.distance}  parent = {v.parent}")

S: distance = 0  parent = None
A: distance = 8  parent = C
B: distance = 9  parent = A
C: distance = 5  parent = S
D: distance = 7  parent = C


In [227]:
g2 = Graph()
for key in ["A", "B", "C", "D"]:
    g2.add_vertex(key)

g2.add_edge("A", "B", 1)
g2.add_edge("A", "C", 4)
g2.add_edge("B", "C", 2)
g2.add_edge("B", "D", 5)
g2.add_edge("C", "D", 1)

dijkstra(g2, "A")

for key, v in g2.vertices.items():
        print(f"{key}: distance = {v.distance}  parent = {v.parent}")

A: distance = 0  parent = None
B: distance = 1  parent = A
C: distance = 3  parent = B
D: distance = 4  parent = C


## Bellman Ford's Algorithm

In [228]:
def bellman_ford(graph, source):
  for key in graph.vertices:
    u = graph.vertices[key]
    u.distance = float('inf')

  graph.vertices[source].distance = 0
  for i in range(len(graph.vertices)-1):
    for j in graph.adj_list:
      u_node = graph.vertices[j]

      for k in graph.adj_list[u_node.key]:
        v_node, v_weight = k[0], k[1]
        if u_node.distance + v_weight < v_node.distance:
          v_node.distance = u_node.distance + v_weight
          v_node.parent = u_node.key

  for j in graph.adj_list:
    u_node = graph.vertices[j]
    for k in graph.adj_list[u_node.key]:
      v_node, v_weight = k[0], k[1]
      if u_node.distance + v_weight < v_node.distance:
        print(u_node.key, v_node.key)
        raise ValueError("Graph contains negative weight cycle")


In [233]:
g = Graph()
for key in ["S", "A", "B", "C", "D"]:
  g.add_vertex(key)

g.add_edge("S", "A", 6)
g.add_edge("S", "C", 7)

g.add_edge("A", "B", 5)
g.add_edge("A", "C", 8)
g.add_edge("A", "D", -4)

g.add_edge("B", "A", -2)

g.add_edge("C", "B", -3)
g.add_edge("C", "D", 9)

g.add_edge("D", "B", 7)
g.add_edge("D", "S", 2)

bellman_ford(g, "S")

In [234]:
for key, v in g.vertices.items():
        print(f"{key}: distance = {v.distance}  parent = {v.parent}")

S: distance = 0  parent = None
A: distance = 2  parent = B
B: distance = 4  parent = C
C: distance = 7  parent = S
D: distance = -2  parent = A
